# Step 2: Data Generation

Generate synthetic EMR data for training.

## Risk Level Calculation

| Score | Risk Level |
|-------|------------|
| 0-2 | LOW |
| 3-5 | MEDIUM |
| 6-8 | HIGH |
| 9+ | CRITICAL |

## Prerequisites

- Run **01_setup_infrastructure.ipynb** first

## Imports and Configuration

In [1]:
%cd ../..
%load_ext autoreload

/Users/ccaudill/src/github/snowflake-ml-prod


In [2]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

2026-04-13 16:46:25,828 - INFO - AST state has not been set explicitly. Defaulting to ast_enabled = True.
2026-04-13 16:46:25,945 - INFO - Snowflake Connector for Python Version: 4.3.0, Python Version: 3.11.14, Platform: macOS-26.4-arm64-arm-64bit
2026-04-13 16:46:25,946 - INFO - Connecting to GLOBAL Snowflake domain


Creating Session...


2026-04-13 16:46:28,782 - INFO - Snowpark Session information: 
"version" : 1.47.0,
"python.version" : 3.11.14,
"python.connector.version" : 4.3.0,
"python.connector.session.id" : 5727214344382730,
"os.name" : Darwin



Connected as: "CCAUDILL"
Current role: "SYSADMIN"
Current warehouse: "ML_DEMO_WAREHOUSE"


## Generate Synthetic Patient Data

The `HistoricalDataGenerator` creates realistic synthetic patient records with correlated clinical features. It performs the following steps:

1. **Generate Raw Data** (`RAW_PATIENT_DATA`) - Creates patient records with:
   - Demographics (age, gender, BMI)
   - Vital signs (heart rate, blood pressure, temperature, respiratory rate, oxygen saturation)
   - Lab values (glucose, creatinine, hemoglobin, WBC count)
   - Clinical context (diagnosis, comorbidities, admission type, insurance)
   - Risk level labels (LOW, MEDIUM, HIGH, CRITICAL) calculated from clinical indicators

2. **Create Baseline Sample** (`BASELINE_PATIENT_DATA`) - A 20% random sample used for:
   - Data drift detection
   - Monitoring distribution shifts over time

3. **Create Test Split** (`TEST_PATIENT_DATA`) - A 20% holdout set used for:
   - Model evaluation after training
   - Promotion criteria validation

**Expected Risk Distribution**: ~40% LOW, ~35% MEDIUM, ~18% HIGH, ~7% CRITICAL

In [4]:
%autoreload
from data.historical import HistoricalDataGenerator

generator = HistoricalDataGenerator(
    session=session,
    database=config.snowflake.database,
    schema_name=config.snowflake.schema_name,
)

NUM_RECORDS = 10000

data_result = generator.run(
    num_records=NUM_RECORDS,
    table_name=config.tables.raw_data,
    create_baseline=True,
    create_test_split=False,
)

print(f"\nData generation complete:")
print(f"  Main table: {data_result['main_table']}")
print(f"  Baseline table: {data_result['baseline_table']}")

2026-04-13 16:46:53,883 - INFO - Running historical data generation: 10000 records
2026-04-13 16:46:53,883 - INFO - Generating 10000 patient records


2026-04-13 16:46:54,644 - INFO - Generated 10000/10000 records
2026-04-13 16:46:54,665 - INFO - Risk level distribution:
RISK_LEVEL
MEDIUM      0.3560
LOW         0.3099
HIGH        0.2146
CRITICAL    0.1195
Name: proportion, dtype: float64
2026-04-13 16:46:54,666 - INFO - Loading 10000 records to ML_DEMO_PIPELINE_DB.HEALTHCARE.RAW_PATIENT_DATA
2026-04-13 16:47:04,655 - INFO - Loaded 10000 records to ML_DEMO_PIPELINE_DB.HEALTHCARE.RAW_PATIENT_DATA
2026-04-13 16:47:04,656 - INFO - Creating baseline sample from ML_DEMO_PIPELINE_DB.HEALTHCARE.RAW_PATIENT_DATA to ML_DEMO_PIPELINE_DB.HEALTHCARE.BASELINE_PATIENT_DATA
2026-04-13 16:47:06,298 - INFO - Created baseline with 1980 records
2026-04-13 16:47:06,300 - INFO - Historical data generation complete



Data generation complete:
  Main table: {'table': 'ML_DEMO_PIPELINE_DB.HEALTHCARE.RAW_PATIENT_DATA', 'records': 10000}
  Baseline table: {'table': 'ML_DEMO_PIPELINE_DB.HEALTHCARE.BASELINE_PATIENT_DATA', 'records': 1980}


## Explore Generated Data

In [5]:
df = session.table(config.full_raw_table).to_pandas()

print(f"Dataset shape: {df.shape}")
print(f"\nRisk Level Distribution:")
print(df["RISK_LEVEL"].value_counts(normalize=True).round(3))

print(f"\nSample Records:")
df[["PATIENT_ID", "AGE", "GENDER", "HEART_RATE", "SYSTOLIC_BP", "OXYGEN_SATURATION", 
    "GLUCOSE_LEVEL", "CREATININE", "COMORBIDITY_COUNT", "INSURANCE_TYPE", "RISK_LEVEL"]].head(10)

Dataset shape: (10000, 23)

Risk Level Distribution:
RISK_LEVEL
MEDIUM      0.356
LOW         0.310
HIGH        0.215
CRITICAL    0.120
Name: proportion, dtype: float64

Sample Records:


,PATIENT_ID,AGE,GENDER,HEART_RATE,SYSTOLIC_BP,OXYGEN_SATURATION,GLUCOSE_LEVEL,CREATININE,COMORBIDITY_COUNT,INSURANCE_TYPE,RISK_LEVEL
0,P8240DA54,50,M,73,122,97.9,93.8,1.02,1,Private,LOW
1,P50CF8FDE,48,M,69,126,98.3,103.9,1.22,1,Medicaid,LOW
2,PFD141B1E,63,F,89,143,94.1,145.0,1.95,5,Medicare,HIGH
3,PA28D7EB6,35,F,74,118,97.9,104.9,0.94,4,Medicaid,MEDIUM
4,PD297F9F9,40,M,90,138,92.3,191.3,2.81,2,Private,HIGH
5,P0D30B941,78,F,76,140,98.6,88.3,1.05,1,Uninsured,MEDIUM
6,P44D45D57,39,M,102,154,90.1,208.5,3.17,9,Private,CRITICAL
7,P237567BD,63,M,77,133,97.7,94.5,1.02,1,Private,LOW
8,P02DB33CD,54,F,92,154,94.4,166.9,2.40,2,Private,HIGH
9,PE6D94EF1,78,Other,76,137,98.4,82.4,0.90,2,Private,MEDIUM


## Next Step

Continue to **03_preprocessing.ipynb**